In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
# 分布確認
# import pandas_profiling as pdp → import ydata_profiling as pdp
# 可視化
import matplotlib.pyplot as plt
# 分布確認
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
# モデリング
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# !pip install japanize-matplotlib
# import japanize_matplotlib
%matplotlib inline

In [ ]:
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df_train.head()

In [ ]:
x_train, y_train, id_train = df_train[['Pclass', 'Fare']], df_train[['Survived']], df_train[['PassengerId']]
print(x_train.shape, y_train.shape, id_train.shape)

In [ ]:
params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.1,
    'num_leaves': 16,
    'n_estimators': 100000,
    'random_state': 123,
    'importance_type': 'gain',
}

def train_cv(input_x, input_y, input_id, params, n_splits=5):
    metrics = []
    imp = pd.DataFrame()

    n_splits = 5
    cv = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123).split(x_train, y_train))

    for nfold in np.arange(n_splits):
        print('-'*20, nfold, '-'* 20)
        idx_tr, idx_va = cv[nfold][0], cv[nfold][1]
        x_tr, y_tr = x_train.loc[idx_tr, :], y_train.loc[idx_tr, :]
        x_va, y_va = x_train.loc[idx_va, :], y_train.loc[idx_va, :]
        print(x_tr.shape, y_tr.shape)
        print(x_va.shape, y_va.shape)
        print('y_train:{:.3f}, y_tr:{:.3f}, y_va:{:.3f}'.format(
            y_train['Survived'].mean(),
            y_tr['Survived'].mean(),
            y_va['Survived'].mean(),
        ))

        model = lgb.LGBMClassifier(**params)
        model.fit(x_tr, y_tr, 
                  eval_set=[(x_tr, y_tr), (x_va, y_va)],
                  callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True)],
                 )

        y_tr_pred = model.predict(x_tr)
        y_va_pred = model.predict(x_va)
        metric_tr = accuracy_score(y_tr, y_tr_pred)
        metric_va = accuracy_score(y_va, y_va_pred)
        print('[accuracy] tr: {:.2f}, va: {:.2f}'.format(metric_tr, metric_va))
        metrics.append([nfold, metric_tr, metric_va])

        _imp = pd.DataFrame({'col': x_train.columns, 'imp': model.feature_importances_, 'nfold': nfold})
        imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

    print('-'*20, 'result', '-'*20)
    metrics = np.array(metrics)
    print(metrics)
    
    print('[cv ] tr: {:.2f}+-{:.2f}, va: {:.2f}+-{:.2f}'.format(
        metrics[:, 1].mean(), metrics[:, 1].std(),
        metrics[:, 2].mean(), metrics[:, 2].std(),
    ))

    imp = imp.groupby('col')['imp'].agg(['mean', 'std'])
    imp.columns = ['imp', 'imp.std']
    imp = imp.reset_index(drop=False)
    
    print('Done.')

    return imp, metrics

In [ ]:
imp, metrics = train_cv(x_train, y_train, id_train, params, n_splits=5)

In [ ]:
x_train, y_train, id_train = df_train[['Pclass', 'Fare', 'Age']], df_train[['Survived']], df_train[['PassengerId']]
print(x_train.shape, y_train.shape, id_train.shape)

In [ ]:
imp, metrics = train_cv(x_train, y_train, id_train, params, n_splits=5)

In [ ]:
imp.sort_values('imp', ascending=False, ignore_index=True)

In [ ]:
df_train.isnull().sum()

In [ ]:
import ydata_profiling as pdp

In [ ]:
df_train.describe().T

In [ ]:
df_train.describe(exclude='number').T

In [ ]:
df_train[['Fare']].agg(['mean', 'std', 'min', 'max']).T

In [ ]:
df_train['Sex'].value_counts()

In [ ]:
pdp.ProfileReport(df_train)

In [ ]:
df_train.isnull().sum()

In [ ]:
(df_train=='').sum()

In [ ]:
df_train['Age_fillna_0'] = df_train['Age'].fillna(0)
df_train.loc[df_train['Age'].isnull(), ['Age', 'Age_fillna_0']].head()

In [ ]:
df_train['Age_fillna_mean'] = df_train['Age'].fillna(df_train['Age'].mean())
df_train.loc[df_train['Age'].isnull(), ['Age', 'Age_fillna_mean']].head()

In [ ]:
df_train['Cabin_fillna_space'] = df_train['Cabin'].fillna('')
df_train.loc[df_train['Cabin'].isnull(), ['Cabin', 'Cabin_fillna_space']]

In [ ]:
df_train['Cabin_fillna_mode'] = df_train['Cabin'].fillna(df_train['Cabin'].mode()[0])
df_train.loc[df_train['Cabin'].isnull(), ['Cabin', 'Cabin_fillna_mode']].head()

In [ ]:
df_train['Age'].agg(['min', 'max'])

In [ ]:
df_train['Age'].hist()

In [ ]:
quartile = df_train['Age'].quantile(q=0.75) - df_train['Age'].quantile(q=0.25)
print('四分位範囲:', quartile)
print('下限値:', df_train['Age'].quantile(q=0.25) - quartile*1.5)
print('上限値:', df_train['Age'].quantile(q=0.75) + quartile*1.5)

In [ ]:
std = StandardScaler()
std.fit(df_train[['Fare']])
print('mean:', std.mean_[0], ', std:', np.sqrt(std.var_[0]))

df_train['Fare_standard'] = std.transform(df_train[['Fare']])
df_train[['Fare', 'Fare_standard']].head()

In [ ]:
mms = MinMaxScaler(feature_range=(0,1))
mms.fit(df_train[['Fare']])
print('min:', mms.data_min_[0], ', max:', mms.data_max_[0])

df_train['Fare_normalize'] = mms.transform(df_train[['Fare']])
df_train[['Fare', 'Fare_normalize']].head()

In [ ]:
df_train['Fare_log'] = np.log(df_train['Fare'] + 1e-5)
df_train[['Fare', 'Fare_log']].head()

In [ ]:
df_train['Fare_square'] = df_train['Fare'].apply(lambda x: x**2)
df_train['Fare_exp'] = df_train['Fare'].apply(lambda x: np.exp(x))
df_train['Fare_reciprocal'] = df_train['Fare'].apply(lambda x: 1/(x+1e-3))
df_train[['Fare', 'Fare_square', 'Fare_exp', 'Fare_reciprocal']].head()

In [ ]:
df_train['Age_bin'] = pd.cut(df_train['Age'],
                            bins=[0,10,20,30,40,50,100],
                            right=False,
                            labels=['10代未満','10代','20代','30代','40代','50代以上'],
                            duplicates='raise',
                            include_lowest=True)
df_train['Age_bin'] = df_train['Age_bin'].astype(str)
df_train[['Age', 'Age_bin']].head()

In [ ]:
df_train['Age_na'] = df_train['Age'].isnull()*1
df_train[['Age', 'Age_na']].head(7)

In [ ]:
df_ohe = pd.get_dummies(df_train[['Embarked', 'Sex']], dummy_na=True, drop_first=False)
df_ohe.head()

In [ ]:
ce_Embarked = df_train['Embarked'].value_counts().to_dict()
print(ce_Embarked)
df_train['Embarked_ce'] = df_train['Embarked'].map(ce_Embarked)
df_train[['Embarked', 'Embarked_ce']].head()

In [ ]:
le_embarked = LabelEncoder()
le_embarked.fit(df_train['Embarked'])
df_train['Embarked_le'] = le_embarked.transform(df_train['Embarked'])
df_train[['Embarked', 'Embarked_le']]

In [ ]:
df_train['Embarked_na'] = df_train['Embarked'].isnull()*1
df_train.loc[df_train['Embarked'].isnull(), ['Embarked', 'Embarked_na']].head()

In [ ]:
df_train['SibSp_+_Parch'] = df_train['SibSp'] + df_train['Parch']
df_train[['SibSp', 'Parch', 'SibSp_+_Parch']].head()

In [ ]:
df_agg = df_train.groupby('Sex')['Fare'].agg(['mean']).reset_index()
df_agg.columns = ['Sex', 'mean_Fare_by_Sex']
print('集約テーブル')
display(df_agg)

df_train = pd.merge(df_train, df_agg, on='Sex', how='left')
print('結合後テーブル')
display(df_train[['Sex', 'Fare', 'mean_Fare_by_Sex']].head())

In [ ]:
df_train['count_Sex_x_Embarked'] = df_train.groupby(['Sex', 'Embarked'])['PassengerId'].transform('count')
df_train[['Sex', 'Embarked', 'count_Sex_x_Embarked']].head()

In [ ]:
df_tbl = pd.crosstab(df_train['Sex'], df_train['Embarked'], normalize='index')
print('集約テーブル（行方向の和で割る）')
display(df_tbl)

df_tbl = df_tbl.reset_index()
df_tbl = pd.melt(df_tbl, id_vars='Sex', value_name='rate_Sex_x_Embarked')
print('集約テーブル（縦持ち変換後）')
display(df_tbl)
, 
df_train = pd.merge(df_train, df_tbl, on=['Sex', 'Embarked'], how='left')
print('結合後テーブル')
df_train[['Sex', 'Embarked', 'rate_Sex_x_Embarked']].head()

In [ ]:
df_train['Sex=male_&_Embarked=S'] = np.where((df_train['Sex']=='male') & (df_train['Embarked']=='S'), 1, 0)
df_train[['Sex', 'Embarked', 'Sex=male_&_Embarked=S']].head()

In [ ]:
# 時系列データ
df1 = pd.DataFrame({'date':pd.date_range('2021-01-01','2021-01-10'),
                   'weather':['晴れ','晴れ','雨','くもり','くもり','晴れ','雨','晴れ','晴れ','晴れ']})
df1['weather_shift1'] = df1['weather'].shift(1)
df1

In [ ]:
df1['weather_shift1'] = df1['weather_shift1'].interpolate(method='bfill')
df1

In [ ]:
df2 = pd.DataFrame({'id':['A']*3 + ['B']*2 + ['C']*4,
                    'date':['2021-04-02','2021-04-10','2021-04-25',
                            '2021-04-18','2021-04-09',
                            '2021-04-01','2021-04-04','2021-04-09','2021-04-12'],
                    'money':[1000,2000,900,4000,1800,900,1200,1100,2900]})
df2['date'] = pd.to_datetime(df2['date'], format='%Y-%m-%d')
df2['money_shift1'] = df2.groupby('id')['money'].shift(1)
df2

In [ ]:
df2['date_shift1'] = df2.groupby('id')['date'].shift(1)
df2['days_elapsed'] = df2['date'] - df2['date_shift1']
df2['days_elapsed'] = df2['days_elapsed'].dt.days
df2

In [ ]:
df3 = pd.DataFrame({'date':pd.date_range('2021-01-01','2021-01-10'),
                    'temperature':[8,10,21,11,9,10,12,7,9,10]})
df3['temperature_window3'] = df3['temperature'].rolling(window=3).mean()
df3

In [ ]:
df5 = pd.DataFrame({'date':pd.date_range('2021-01-01','2021-01-10'),
                    'flag_rain':[0,0,1,0,0,0,1,0,0,0]})
df5['flag_rain_cumsum'] = df5['flag_rain'].cumsum()
df5

In [ ]:
df6 = pd.DataFrame({'id':['A']*3 + ['B']*2 + ['C']*4,
                    'date':['2021-04-02','2021-04-10','2021-04-25',
                            '2021-04-18','2021-04-09',
                            '2021-04-01','2021-04-04','2021-04-09','2021-04-12'],
                    'money':[1000,2000,900,4000,1800,900,1200,1100,2900]})
df6['date'] = pd.to_datetime(df6['date'], format='%Y-%m-%d')
df6['money_cumsum'] = df6.groupby('id')['money'].cumsum()
df6

In [ ]:
# テキストデータ
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
vec = CountVectorizer(min_df=20)

vec.fit(df_train['Name'])

df_name = pd.DataFrame(vec.transform(df_train['Name']).toarray(), columns=vec.get_feature_names_out())
print(df_name.shape)
df_name.head()